In [7]:
from skmultilearn.dataset import load_from_arff
import numpy as np

# ---- Train ----
X_train, y_train = load_from_arff(
    "plantgo-train.arff",
    label_count=12,        # PlantGO has 12 labels
    label_location="end",
    load_sparse=True
)

# ---- Test ----
X_test, y_test = load_from_arff(
    "PlantGO-test.arff",
    label_count=12,
    label_location="end",
    load_sparse=True
)

# Convert sparse → dense (if your SVM needs NumPy)
X_train = X_train.toarray().astype(np.float32)
X_test = X_test.toarray().astype(np.float32)

y_train = y_train.toarray().astype(np.int32)
y_test = y_test.toarray().astype(np.int32)

print("Train X:", X_train.shape)
print("Train y:", y_train.shape)
print("Test X:", X_test.shape)
print("Test y:", y_test.shape)

# Save
np.save("X_train.npy", X_train)
np.save("y_train.npy", y_train)
np.save("X_test.npy", X_test)
np.save("y_test.npy", y_test)

Train X: (656, 3091)
Train y: (656, 12)
Test X: (322, 3091)
Test y: (322, 12)


##### SVM Multi Label Algorithm

Training/Testing Pseudocode procedure TRAIN_BINARY_SVM(X, y, learning_rate, lambda_param, n_iters): # y is in {0,1} y_mapped ← map y to {-1,+1}: y_mapped[i] = -1 if y[i] ≤ 0 else +1

(n_samples, n_features) ← shape(X)
w ← zero vector of length n_features
b ← 0

for iter in 1..n_iters:
    for i in 1..n_samples:
        x_i ← X[i]
        y_i ← y_mapped[i]
        margin_ok ← ( y_i * ( dot(x_i, w) + b ) ) ≥ 1
        if margin_ok:
            # only regularization update
            w ← w - learning_rate * (2 * lambda_param * w)
        else:
            # hinge-loss + regularization update
            w ← w - learning_rate * (2 * lambda_param * w - y_i * x_i)
            b ← b - learning_rate * y_i

return (w, b)
procedure TRAIN_MULTILABEL_SVM(X, Y, svm_params): # Y is binary indicator matrix in {0,1}, shape (n_samples, n_labels) L ← number of labels = number of columns in Y classifiers ← empty map/dictionary

for label_idx in 1..L:
    y_label ← Y[:, label_idx]              # one column (binary)
    (w, b) ← TRAIN_BINARY_SVM(X, y_label, svm_params)
    classifiers[label_idx] ← (w, b)

return classifiers
procedure PREDICT_BINARY_SVM(X, w, b): # outputs labels in {0,1} scores ← X · w + b # vector length n_samples y_pred ← 1 where scores ≥ 0 else 0 return y_pred

procedure PREDICT_MULTILABEL_SVM(X, classifiers): L ← number of classifiers predictions_list ← empty list

for label_idx in 1..L:
    (w, b) ← classifiers[label_idx]
    preds_label ← PREDICT_BINARY_SVM(X, w, b)   # shape (n_samples,)
    append preds_label to predictions_list

Ŷ ← transpose(stack(predictions_list))          # shape (n_samples, L)
return Ŷ

In [8]:
class BinaryLinearSVM:
    """
    Linear SVM trained with mini-batch SGD on:
        lambda * ||w||^2 + mean(max(0, 1 - y*(x·w + b)))
    where y in {-1, +1}.
    """
    def __init__(self, lr=0.05, lambda_param=1e-4, n_epochs=30, batch_size=128, seed=42):
        self.lr = float(lr)
        self.lambda_param = float(lambda_param)
        self.n_epochs = int(n_epochs)
        self.batch_size = int(batch_size)
        self.seed = int(seed)
        self.w = None
        self.b = None

    def fit(self, X, y01):
        # y01 in {0,1} -> y in {-1,+1}
        y = np.where(y01 > 0, 1.0, -1.0).astype(np.float32)

        n, d = X.shape
        self.w = np.zeros(d, dtype=np.float32)
        self.b = 0.0

        rng = np.random.default_rng(self.seed)

        for epoch in range(self.n_epochs):
            idx = rng.permutation(n)

            for start in range(0, n, self.batch_size):
                batch_idx = idx[start:start + self.batch_size]
                xb = X[batch_idx]
                yb = y[batch_idx]

                scores = xb @ self.w + self.b                 # (B,)
                margins = yb * scores                          # (B,)
                active = margins < 1.0                         # hinge active

                # Gradients (mean over batch)
                # reg part: 2*lambda*w
                grad_w = 2.0 * self.lambda_param * self.w

                if np.any(active):
                    xa = xb[active]
                    ya = yb[active]
                    # hinge part: -(1/B) * sum( y_i * x_i )
                    grad_w -= (ya[:, None] * xa).mean(axis=0)
                    # grad_b = -(1/B) * sum(y_i)
                    grad_b = -ya.mean()
                else:
                    grad_b = 0.0

                # Update: w <- w - lr*grad_w ; b <- b - lr*grad_b
                self.w -= self.lr * grad_w
                self.b -= self.lr * grad_b

        return self

    def decision_function(self, X):
        return X @ self.w + self.b  # real-valued scores

    def predict(self, X, threshold=0.0):
        return (self.decision_function(X) >= threshold).astype(np.int32)


class MultiLabelLinearSVM:
    """
    One-vs-Rest multi-label classifier:
    trains one BinaryLinearSVM per label column.

    threshold_strategy:
      - "zero": threshold=0 for all labels
      - "prevalence": choose threshold per label so that #positives on train
                      matches the label prevalence in training
    """
    def __init__(self, svm_params=None, threshold_strategy="prevalence"):
        self.svm_params = svm_params or {}
        self.threshold_strategy = threshold_strategy
        self.classifiers = []
        self.thresholds_ = None
        self.n_labels_ = None

    def fit(self, X, Y):
        X = np.asarray(X, dtype=np.float32)
        Y = np.asarray(Y)

        n, d = X.shape
        assert Y.ndim == 2 and Y.shape[0] == n, "Y must be (n_samples, n_labels)"
        self.n_labels_ = Y.shape[1]

        self.classifiers = []
        self.thresholds_ = np.zeros(self.n_labels_, dtype=np.float32)

        for j in range(self.n_labels_):
            yj = Y[:, j].astype(np.int32)

            clf = BinaryLinearSVM(**self.svm_params)
            clf.fit(X, yj)
            self.classifiers.append(clf)

            # --- choose threshold ---
            if self.threshold_strategy == "zero":
                self.thresholds_[j] = 0.0

            elif self.threshold_strategy == "prevalence":
                # Match number of predicted positives to training positives
                scores = clf.decision_function(X)
                k = int(yj.sum())
                if k <= 0:
                    self.thresholds_[j] = np.inf   # never predict positive
                elif k >= n:
                    self.thresholds_[j] = -np.inf  # always positive
                else:
                    # threshold is the k-th largest score
                    # so that exactly k samples are predicted positive on train
                    self.thresholds_[j] = np.partition(scores, n - k)[n - k]
            else:
                raise ValueError("threshold_strategy must be 'zero' or 'prevalence'")

        return self

    def decision_function(self, X):
        X = np.asarray(X, dtype=np.float32)
        scores = np.zeros((X.shape[0], self.n_labels_), dtype=np.float32)
        for j, clf in enumerate(self.classifiers):
            scores[:, j] = clf.decision_function(X)
        return scores

    def predict(self, X):
        scores = self.decision_function(X)
        return (scores >= self.thresholds_[None, :]).astype(np.int32)


##### Evaluation Metrics

1. Hamming Loss: It meansures the fration of label decisions that are wrong (averaged over all samples × labels)
It is a good indicator because it evaluates each label independently (robust when exact match is rare).

2. Micro-averaged Precision / Recall / F1: It aggregates TP/FP/FN across all labels, and emphasizes performance on frequent labels and overall correctness.

3. Macro-averaged F1
Compute F1 per label, then average → treats each label equally.
Best when you care about rare labels too.

4. Subset Accuracy (Exact Match)
Strict: counts a sample correct only if all labels match.
Useful but often low; include as a “hard” metric to show exact multi-label correctness.

In [9]:
def multilabel_confusion(Y_true, Y_pred):
    # Y_* shape: (n_samples, n_labels) with {0,1}
    tp = np.sum((Y_true == 1) & (Y_pred == 1))
    fp = np.sum((Y_true == 0) & (Y_pred == 1))
    fn = np.sum((Y_true == 1) & (Y_pred == 0))
    tn = np.sum((Y_true == 0) & (Y_pred == 0))
    return tp, fp, fn, tn

def micro_precision_recall_f1(Y_true, Y_pred, eps=1e-12):
    tp, fp, fn, _ = multilabel_confusion(Y_true, Y_pred)
    prec = tp / (tp + fp + eps)
    rec  = tp / (tp + fn + eps)
    f1   = 2 * prec * rec / (prec + rec + eps)
    return prec, rec, f1

def macro_f1(Y_true, Y_pred, eps=1e-12):
    # average F1 across labels
    L = Y_true.shape[1]
    f1s = []
    for l in range(L):
        yt, yp = Y_true[:, l], Y_pred[:, l]
        tp = np.sum((yt==1) & (yp==1))
        fp = np.sum((yt==0) & (yp==1))
        fn = np.sum((yt==1) & (yp==0))
        prec = tp/(tp+fp+eps)
        rec  = tp/(tp+fn+eps)
        f1   = 2*prec*rec/(prec+rec+eps)
        f1s.append(f1)
    return float(np.mean(f1s))

def hamming_loss(Y_true, Y_pred):
    return float(np.mean(Y_true != Y_pred))

def subset_accuracy(Y_true, Y_pred):
    return float(np.mean(np.all(Y_true == Y_pred, axis=1)))

##### Implement and Results


In [17]:
def train_test_report(X_train, Y_train, X_test, Y_test, svm_params, threshold_strategy="prevalence"):
    model = MultiLabelLinearSVM(svm_params=svm_params, threshold_strategy=threshold_strategy)
    model.fit(X_train, Y_train)

    Y_pred_train = model.predict(X_train)
    Y_pred_test  = model.predict(X_test)

    def pack_metrics(Yt, Yp):
        p, r, f1 = micro_precision_recall_f1(Yt, Yp)
        return {
            "HammingLoss": hamming_loss(Yt, Yp),
            "SubsetAcc": subset_accuracy(Yt, Yp),
            "MicroP": p, "MicroR": r, "MicroF1": f1,
            "MacroF1": macro_f1(Yt, Yp),
        }

    train_m = pack_metrics(Y_train, Y_pred_train)
    test_m  = pack_metrics(Y_test,  Y_pred_test)

    return train_m, test_m

svm_params = {"lr": 0.05, "lambda_param": 1e-4, "n_epochs": 40, "batch_size": 256, "seed": 42}
train_m, test_m = train_test_report(np.load("X_train.npy"), np.load("y_train.npy"), np.load("X_test.npy"), np.load("y_test.npy"), svm_params)

print("Training Metrics:")
for k, v in train_m.items():
    print(f"  {k}: {v}")

print("\nTesting Metrics:")
for k, v in test_m.items():
    print(f"  {k}: {v}")

Training Metrics:
  HammingLoss: 0.010797764227642276
  SubsetAcc: 0.885670731707317
  MicroP: 0.9384615384615371
  MicroR: 0.9424157303370773
  MicroF1: 0.9404344779252168
  MacroF1: 0.9435913358631759

Testing Metrics:
  HammingLoss: 0.036231884057971016
  SubsetAcc: 0.6894409937888198
  MicroP: 0.8103975535168171
  MicroR: 0.7725947521865866
  MicroF1: 0.7910447761189008
  MacroF1: 0.6572695306025029


##### SVM Multi Class Algorithm

##### Dataset - MNIST: Modified National Institute of Standards and Technology database
It contains: 70,000 grayscale images; Image size: 28 * 28 pixels; 10 classes (digits 0-9); 60,000 training images; 10,000 test images.

In [ ]:
from sklearn.datasets import fetch_openml
mnist = fetch_openml('mnist_784', version=1)
X = mnist.data
y = mnist.target.astype(int)

print("MNIST X:", X.shape)
print("MNIST y:", y.shape)

MNIST X: (70000, 784)
MNIST y: (70000,)


##### Pseudocode